## Robot Prediction Comparison

Compare one offline prediction JSONL against one line from the deployment prediction JSONL.

In [1]:
from pathlib import Path
import json

import numpy as np

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent


def repo_path(path):
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path


# Inputs to change.
OFFLINE_JSONL_PATH = repo_path("data/debug_grasp_cube_52/offline_pred_000000.jsonl")
PREDICTIONS_JSONL_PATH = repo_path("data/debug_grasp_cube_52/pred_only_log_20260522_145223.jsonl")
PREDICTIONS_LINE_NUMBER = 1  # 1-based line number in PREDICTIONS_JSONL_PATH
OFFLINE_LINE_NUMBER = 1      # 1-based; offline_pred files normally have one line
ROBOT_IDX = 0


def load_jsonl_line(path, line_number):
    line_index = line_number - 1
    with Path(path).open("r") as f:
        for idx, line in enumerate(f):
            if idx == line_index:
                return json.loads(line)
    raise IndexError(f"{path} has no line_number={line_number}")


def robot_horizon(record, robot_idx):
    current_pose = np.asarray(record["current_pose"], dtype=np.float64)
    num_robots = current_pose.shape[0]
    horizon = np.asarray(record["full_horizon_prediction"], dtype=np.float64)
    horizon = horizon.reshape(horizon.shape[0], num_robots, 7)
    return current_pose[robot_idx], horizon[:, robot_idx, :]


offline_record = load_jsonl_line(OFFLINE_JSONL_PATH, OFFLINE_LINE_NUMBER)
prediction_record = load_jsonl_line(PREDICTIONS_JSONL_PATH, PREDICTIONS_LINE_NUMBER)

current_pose, prediction_horizon = robot_horizon(prediction_record, ROBOT_IDX)
_, offline_horizon = robot_horizon(offline_record, ROBOT_IDX)

diff = offline_horizon - prediction_horizon
pos_diff = np.linalg.norm(diff[:, :3], axis=1)
rot_diff = np.linalg.norm(diff[:, 3:6], axis=1)
gripper_diff = np.abs(diff[:, 6])

print("offline jsonl:", OFFLINE_JSONL_PATH)
print("prediction jsonl:", PREDICTIONS_JSONL_PATH)
print("prediction line number:", PREDICTIONS_LINE_NUMBER)
print("offline horizon shape:", offline_horizon.shape)
print("prediction horizon shape:", prediction_horizon.shape)
print("avg position difference (m):", float(pos_diff.mean()))
print("avg rotation-vector difference:", float(rot_diff.mean()))
print("avg gripper difference (m):", float(gripper_diff.mean()))

offline jsonl: /Users/yifanluo/Robotouch Lab/universal_manipulation_interface/data/debug_grasp_cube_52/offline_pred_000000.jsonl
prediction jsonl: /Users/yifanluo/Robotouch Lab/universal_manipulation_interface/data/debug_grasp_cube_52/pred_only_log_20260522_145223.jsonl
prediction line number: 1
offline horizon shape: (16, 7)
prediction horizon shape: (16, 7)
avg position difference (m): 0.02962236954046212
avg rotation-vector difference: 0.05998093144037205
avg gripper difference (m): 0.0027336319908499718


In [2]:
import vedo
from scipy.spatial.transform import Rotation as R


def axis_actors(xyz, rotvec, scale=0.025, alpha=0.8):
    rot = R.from_rotvec(rotvec).as_matrix()
    return [
        vedo.Arrows(xyz, xyz + rot[:, :, 0] * scale, c="red", alpha=alpha),
        vedo.Arrows(xyz, xyz + rot[:, :, 1] * scale, c="green", alpha=alpha),
        vedo.Arrows(xyz, xyz + rot[:, :, 2] * scale, c="blue", alpha=alpha),
    ]


current_xyz = current_pose[None, :3]
current_rot = current_pose[None, 3:6]
prediction_xyz = prediction_horizon[:, :3]
prediction_rot = prediction_horizon[:, 3:6]
offline_xyz = offline_horizon[:, :3]
offline_rot = offline_horizon[:, 3:6]

actors = [
    vedo.Points(current_xyz, c="yellow", r=18),
    vedo.Line(np.vstack([current_xyz, prediction_xyz]), c="orange", lw=5),
    vedo.Points(prediction_xyz, c="orange", r=8),
    vedo.Line(np.vstack([current_xyz, offline_xyz]), c="#1f77b4", lw=5),
    vedo.Points(offline_xyz, c="#1f77b4", r=8),
]
actors += axis_actors(current_xyz, current_rot, scale=0.035, alpha=1.0)
actors += axis_actors(prediction_xyz, prediction_rot, scale=0.025, alpha=0.45)
actors += axis_actors(offline_xyz, offline_rot, scale=0.025, alpha=0.9)

print("yellow = current pose")
print("orange = prediction JSONL line")
print("blue = offline prediction JSONL")

vedo.settings.default_backend = "vtk"
vedo.show(
    actors,
    axes=1,
    viewup="z",
    bg="white",
    title=f"Offline vs prediction line {PREDICTIONS_LINE_NUMBER}",
)

yellow = current pose
orange = prediction JSONL line
blue = offline prediction JSONL
